In [ ]:
import json
from pathlib import Path

import pandas as pd

In [31]:
def load_session_to_df(session_data, df):
    """
    Recibe un JSON ya cargado (dict) y agrega sus interacciones
    al DataFrame recibido.

    Parámetros:
        session_data (dict): JSON de la sesión
        df (pd.DataFrame): DataFrame acumulador

    Retorna:
        pd.DataFrame actualizado
    """

    try:
        session_id = session_data.get("session_id")
        user_id = session_data.get("user_id")
        created_at = session_data.get("created_at")
        session_mode = session_data.get("mode")
        session_use_rag = session_data.get("use_rag")

        feedback = session_data.get("feedback", {})
        csat_score = feedback.get("csat", {}).get("score")
        nps_score = feedback.get("nps", {}).get("score")
        nps_category = feedback.get("nps", {}).get("category")
        resolution_label = feedback.get("resolution", {}).get("label")
        resolution_numeric = feedback.get("resolution", {}).get("numeric")

        rows = []

        for idx, interaction in enumerate(session_data.get("interactions", []), start=1):

            titles = interaction.get("titles", [])
            scores = interaction.get("scores", [])

            row = {
                "session_id": session_id,
                "user_id": user_id,
                "created_at": created_at,
                "session_mode": session_mode,
                "session_use_rag": session_use_rag,

                "interaction_number": idx,
                "timestamp": interaction.get("timestamp"),
                "query": interaction.get("query"),
                "routed_query": interaction.get("routed_query"),
                "mode": interaction.get("mode"),
                "use_rag": interaction.get("use_rag"),
                "used_cache": interaction.get("used_cache"),
                "latency_ms": interaction.get("latency"),
                "results_count": interaction.get("results_count"),

                "title_response": titles[0] if len(titles) > 0 else None,
                "score_response": scores[0] if len(scores) > 0 else None,

                "csat_score": csat_score,
                "nps_score": nps_score,
                "nps_category": nps_category,
                "resolution_label": resolution_label,
                "resolution_numeric": resolution_numeric
            }

            rows.append(row)

        temp_df = pd.DataFrame(rows)
        df = pd.concat([df, temp_df], ignore_index=True)

        return df

    except Exception as e:
        print(f"Error procesando sesión: {e}")
        return df

In [32]:
def load_all_sessions(folder_path):
    """
    Lee todos los archivos JSON cuyo nombre inicie con session_
    y consolida la información en un DataFrame.
    """

    folder_path = Path(folder_path)
    df = pd.DataFrame()

    files = list(folder_path.glob("session_*.json"))

    total_files = len(files)
    loaded_files = 0
    failed_files = 0

    print(f"Archivos encontrados: {total_files}\n")

    for i, file in enumerate(files, start=1):
        try:
            with open(file, "r", encoding="utf-8") as f:
                session_data = json.load(f)

            df = load_session_to_df(session_data, df)

            loaded_files += 1
            print(f"[{i}/{total_files}] OK -> {file.name}")

        except Exception as e:
            failed_files += 1
            print(f"[{i}/{total_files}] ERROR -> {file.name}: {e}")

    total_interactions = len(df)

    print("\n========== RESUMEN ==========")
    print(f"Archivos encontrados : {total_files}")
    print(f"Archivos leídos      : {loaded_files}")
    print(f"Archivos fallidos    : {failed_files}")
    print(f"Interacciones totales: {total_interactions}")
    print("=============================")

    return df

In [33]:
# Ruta carpeta origen
folder_path = Path(
    r"C:\Users\juans\OneDrive\Documentos\Maestria en Ingenieria y Analitica de Datos\Proyecto de Grado\CineMate AI\interactions"
)

# Cargar todos los archivos session_*.json
df_sessions = load_all_sessions(folder_path)

# Ver resultados
df_sessions.head()

Archivos encontrados: 11

[1/11] OK -> session_0d2944f1-d1dc-441b-ac4c-d722ea02b38c.json
Error procesando sesión: 'NoneType' object has no attribute 'get'
[2/11] OK -> session_29fa135c-dc1e-4e20-8ef3-c2c900ff6479.json
[3/11] OK -> session_6d5b775c-4fa9-4f61-bd49-a36a0982fd5f.json
[4/11] OK -> session_7251b23c-d3e2-4530-a4a2-8609c257f67d.json
Error procesando sesión: 'NoneType' object has no attribute 'get'
[5/11] OK -> session_821a5308-b4c6-40b1-851a-c9a2b3aa7689.json
[6/11] OK -> session_a5646d83-cec6-48af-93be-c7bde3698897.json
[7/11] OK -> session_b6f21778-3e7d-44a8-a745-6f60e3315a6a.json
[8/11] OK -> session_c4d4f301-ba9c-439d-aa91-7ef9ccc85368.json
Error procesando sesión: 'NoneType' object has no attribute 'get'
[9/11] OK -> session_cc536b46-1093-417d-b4bb-e4bda4d7033a.json
[10/11] OK -> session_d0abd758-bf5f-4dc2-a24c-5823f7476f10.json
[11/11] OK -> session_e49163ac-bffa-4a10-934f-bbf0515d7abf.json

========== RESUMEN ==========
Archivos encontrados : 11
Archivos leídos      : 1

,session_id,user_id,created_at,session_mode,session_use_rag,interaction_number,timestamp,query,routed_query,mode,...,used_cache,latency_ms,results_count,title_response,score_response,csat_score,nps_score,nps_category,resolution_label,resolution_numeric
0,0d2944f1-d1dc-441b-ac4c-d722ea02b38c,573167556912,2026-04-13T19:22:57.927262,RAG,True,1,2026-04-13T19:33:13.537968,recomiendame algo como pacific rim,recomiendame algo como pacific rim,RAG,...,False,613.449,5,Pacific Rim,0.4023,5,10,promoter,yes,3
1,0d2944f1-d1dc-441b-ac4c-d722ea02b38c,573167556912,2026-04-13T19:22:57.927262,RAG,True,2,2026-04-13T19:53:19.798403,ya las vi,ya las vi,RAG,...,False,1065.263,5,The Las Vegas Story,0.3925,5,10,promoter,yes,3
2,6d5b775c-4fa9-4f61-bd49-a36a0982fd5f,573167556912,2026-04-15T22:59:41.391712,DIRECT,False,1,2026-04-15T23:06:30.523632,recomiendame algo parecido a transformers,recomiendame algo parecido a transformers,DIRECT,...,False,405.990,5,Babcock Transformers,0.4141,5,10,promoter,yes,3
3,7251b23c-d3e2-4530-a4a2-8609c257f67d,573167556912,2026-04-16T02:32:12.273956,RAG,True,1,2026-04-16T02:36:17.524085,recomiendame algo como pacific rim,recomiendame algo como pacific rim,RAG,...,False,154.662,5,Pacific Rim,0.4023,5,7,passive,yes,3
4,a5646d83-cec6-48af-93be-c7bde3698897,573167556912,2026-04-15T22:36:30.504774,RAG,True,1,2026-04-15T22:42:02.311397,recomiendame algo parecido a tranformers,recomiendame algo parecido a tranformers,RAG,...,False,327.578,5,Humans,0.1979,5,10,promoter,yes,3


In [34]:
# created_at -> solo fecha
df_sessions["created_at"] = pd.to_datetime(
    df_sessions["created_at"], errors="coerce"
).dt.strftime("%Y-%m-%d")

# timestamp -> solo hora:minuto
df_sessions["timestamp"] = pd.to_datetime(
    df_sessions["timestamp"], errors="coerce"
).dt.strftime("%H:%M")

# latency_ms -> minutos
df_sessions["latency_ms"] = df_sessions["latency_ms"] / 60000


df_sessions["user_id"] = (
    df_sessions["user_id"]
    .astype(str)
    .str.replace(r"^57(\d{3})(\d+)$", r"57 \1 \2", regex=True)
)

df_sessions.head()

,session_id,user_id,created_at,session_mode,session_use_rag,interaction_number,timestamp,query,routed_query,mode,...,used_cache,latency_ms,results_count,title_response,score_response,csat_score,nps_score,nps_category,resolution_label,resolution_numeric
0,0d2944f1-d1dc-441b-ac4c-d722ea02b38c,57 316 7556912,2026-04-13,RAG,True,1,19:33,recomiendame algo como pacific rim,recomiendame algo como pacific rim,RAG,...,False,0.010224,5,Pacific Rim,0.4023,5,10,promoter,yes,3
1,0d2944f1-d1dc-441b-ac4c-d722ea02b38c,57 316 7556912,2026-04-13,RAG,True,2,19:53,ya las vi,ya las vi,RAG,...,False,0.017754,5,The Las Vegas Story,0.3925,5,10,promoter,yes,3
2,6d5b775c-4fa9-4f61-bd49-a36a0982fd5f,57 316 7556912,2026-04-15,DIRECT,False,1,23:06,recomiendame algo parecido a transformers,recomiendame algo parecido a transformers,DIRECT,...,False,0.006766,5,Babcock Transformers,0.4141,5,10,promoter,yes,3
3,7251b23c-d3e2-4530-a4a2-8609c257f67d,57 316 7556912,2026-04-16,RAG,True,1,02:36,recomiendame algo como pacific rim,recomiendame algo como pacific rim,RAG,...,False,0.002578,5,Pacific Rim,0.4023,5,7,passive,yes,3
4,a5646d83-cec6-48af-93be-c7bde3698897,57 316 7556912,2026-04-15,RAG,True,1,22:42,recomiendame algo parecido a tranformers,recomiendame algo parecido a tranformers,RAG,...,False,0.005460,5,Humans,0.1979,5,10,promoter,yes,3


In [35]:
# Ruta destino
output_file = Path(
    r"C:\Users\juans\OneDrive\Documentos\Maestria en Ingenieria y Analitica de Datos\Proyecto de Grado\CineMate AI\data\raw\chatbot_sessions_raw.csv"
)

# Guardar CSV
df_sessions.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Archivo guardado en:")
print(output_file)

Archivo guardado en:
C:\Users\juans\OneDrive\Documentos\Maestria en Ingenieria y Analitica de Datos\Proyecto de Grado\CineMate AI\data\raw\chatbot_sessions_raw.csv
